# Systolic matrix multiplier performance benchmark

This notebook benchmarks the board-validated 16x16 Q8.8 path. Reported latency is **PS-observed end-to-end latency**: AXI-Lite launch, two MM2S transfers, accelerator execution, and one S2MM transfer. It is not the isolated RTL cycle count.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import matplotlib.pyplot as plt
import gc
import time

BITFILE = '/home/xilinx/pynq/overlays/matmul_systolic/design_1.bit'
overlay = Overlay(BITFILE)

# These names match the board-validated axis_test.ipynb design.
dma_a_and_c = overlay.axi_dma_0
dma_b = overlay.axi_dma_1
matmul = overlay.matmul_top_0
print('Overlay loaded:', BITFILE)
print('IPs:', sorted(overlay.ip_dict))

In [ ]:
REG_CTRL, REG_M, REG_N, REG_K = 0x00, 0x10, 0x18, 0x20
M = N = K = 16
ELEMENTS = M * N
OPS_PER_RUN = 2 * M * N * K
BYTES_PER_RUN = 4 * (M*K + K*N + M*N)

def dma_status():
    return {
        'A_MM2S': hex(dma_a_and_c.mmio.read(0x04)),
        'B_MM2S': hex(dma_b.mmio.read(0x04)),
        'C_S2MM': hex(dma_a_and_c.mmio.read(0x34)),
        'HLS_CTRL': hex(matmul.read(REG_CTRL)),
    }

def wait_idle(channel, name, timeout=5.0):
    # Busy-poll deliberately: a 100 us sleep is longer than the expected PL work.
    deadline = time.monotonic() + timeout
    while not channel.idle:
        if time.monotonic() >= deadline:
            raise TimeoutError(f'{name} timeout: {dma_status()}')

print(f'{M}x{K} @ {K}x{N}: {OPS_PER_RUN:,} operations, {BYTES_PER_RUN:,} DMA bytes/run')
print('Initial status:', dma_status())

In [ ]:
rng = np.random.default_rng(42)
A = rng.uniform(-1.0, 1.0, (M, K)).astype(np.float32)
B = rng.uniform(-1.0, 1.0, (K, N)).astype(np.float32)
A_q = np.rint(np.clip(A, -128, 127.99609375) * 256).astype(np.int16)
B_q = np.rint(np.clip(B, -128, 127.99609375) * 256).astype(np.int16)
C_ref = (A_q.astype(np.float32) / 256.0) @ (B_q.astype(np.float32) / 256.0)

a_buf = allocate((ELEMENTS,), dtype=np.int32)
b_buf = allocate((ELEMENTS,), dtype=np.int32)
c_buf = allocate((ELEMENTS,), dtype=np.int32)
np.copyto(a_buf, A_q.astype(np.int32).ravel())
np.copyto(b_buf, B_q.astype(np.int32).ravel())
c_buf[:] = 0

# Dimensions are constant throughout this benchmark; configure them once.
matmul.write(REG_M, M)
matmul.write(REG_N, N)
matmul.write(REG_K, K)
print('Buffers:', {k: hex(v.physical_address) for k, v in {'A': a_buf, 'B': b_buf, 'C': c_buf}.items()})

In [ ]:
def run_once(timeout=5.0):
    c_buf[:] = 0
    t0 = time.perf_counter_ns()
    matmul.write(REG_CTRL, 1)
    t1 = time.perf_counter_ns()

    # Arm receive first so output backpressure cannot block the core.
    dma_a_and_c.recvchannel.transfer(c_buf)
    t2 = time.perf_counter_ns()
    dma_a_and_c.sendchannel.transfer(a_buf)
    t3 = time.perf_counter_ns()
    dma_b.sendchannel.transfer(b_buf)
    t4 = time.perf_counter_ns()
    wait_idle(dma_a_and_c.sendchannel, 'A MM2S', timeout)
    wait_idle(dma_b.sendchannel, 'B MM2S', timeout)
    wait_idle(dma_a_and_c.recvchannel, 'C S2MM', timeout)
    t5 = time.perf_counter_ns()
    return {
        'control_s': (t1 - t0) * 1e-9,
        'c_recv_start_s': (t2 - t1) * 1e-9,
        'a_send_start_s': (t3 - t2) * 1e-9,
        'b_send_start_s': (t4 - t3) * 1e-9,
        'completion_s': (t5 - t4) * 1e-9,
        'total_s': (t5 - t0) * 1e-9,
    }

smoke_timing = run_once()
C_hw = np.asarray(c_buf).reshape(M, N).astype(np.float32) / 256.0
error = np.abs(C_hw - C_ref)
print(f"Smoke test: {smoke_timing['total_s']*1e6:.2f} us, max error={error.max():.6f}, mean error={error.mean():.6f}")
assert error.max() < 0.04, 'Hardware result exceeds Q8.8 tolerance'

In [ ]:
WARMUP_RUNS = 30
MEASURED_RUNS = 100

for _ in range(WARMUP_RUNS):
    run_once()

gc_was_enabled = gc.isenabled()
gc.disable()
try:
    samples = [run_once() for _ in range(MEASURED_RUNS)]
finally:
    if gc_was_enabled:
        gc.enable()
latency_s = np.array([sample['total_s'] for sample in samples])
control_us = np.array([sample['control_s'] for sample in samples]) * 1e6
c_recv_start_us = np.array([sample['c_recv_start_s'] for sample in samples]) * 1e6
a_send_start_us = np.array([sample['a_send_start_s'] for sample in samples]) * 1e6
b_send_start_us = np.array([sample['b_send_start_s'] for sample in samples]) * 1e6
completion_us = np.array([sample['completion_s'] for sample in samples]) * 1e6
latency_us = latency_s * 1e6
gops = OPS_PER_RUN / latency_s / 1e9
dma_mib_s = BYTES_PER_RUN / latency_s / (1024**2)

summary = {
    'runs': MEASURED_RUNS,
    'latency_mean_us': float(latency_us.mean()),
    'latency_median_us': float(np.median(latency_us)),
    'latency_p95_us': float(np.percentile(latency_us, 95)),
    'latency_min_us': float(latency_us.min()),
    'throughput_mean_GOPS': float(gops.mean()),
    'throughput_peak_GOPS': float(gops.max()),
    'effective_DMA_mean_MiB_s': float(dma_mib_s.mean()),
    'control_mean_us': float(control_us.mean()),
    'C_recv_start_mean_us': float(c_recv_start_us.mean()),
    'A_send_start_mean_us': float(a_send_start_us.mean()),
    'B_send_start_mean_us': float(b_send_start_us.mean()),
    'completion_mean_us': float(completion_us.mean()),
}
for key, value in summary.items():
    print(f'{key:28s}: {value:.3f}' if isinstance(value, float) else f'{key:28s}: {value}')

## 连续批次测试

当前设计使用 simple-mode DMA，且每个 tile 都产生一次 TLAST，因此不能用一个 DMA 描述符安全地提交多个 tile。下面在同一个外层计时窗口中连续运行多次，以减少 Jupyter 单元和结果收集的影响；每次运行仍然包含 AXI-Lite 与 DMA 启动开销。

In [ ]:
BATCHES = 10
RUNS_PER_BATCH = 50
batch_latency_s = []
gc_was_enabled = gc.isenabled()
gc.disable()
try:
    for _ in range(BATCHES):
        batch_start = time.perf_counter_ns()
        for _ in range(RUNS_PER_BATCH):
            run_once()
        batch_latency_s.append((time.perf_counter_ns() - batch_start) * 1e-9)
finally:
    if gc_was_enabled:
        gc.enable()

batch_latency_s = np.array(batch_latency_s)
batch_per_run_us = batch_latency_s / RUNS_PER_BATCH * 1e6
batch_gops = OPS_PER_RUN * RUNS_PER_BATCH / batch_latency_s / 1e9
print(f'Batches: {BATCHES} x {RUNS_PER_BATCH} runs')
print(f'Per-run batch median: {np.median(batch_per_run_us):.2f} us')
print(f'Batch throughput mean: {batch_gops.mean():.6f} GOPS')
print(f'Batch vs single median: {(np.median(batch_per_run_us) / np.median(latency_us) - 1) * 100:+.2f}%')
print('Note: simple-mode DMA still restarts for every tile.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].plot(np.arange(1, MEASURED_RUNS + 1), latency_us, linewidth=1)
axes[0, 0].axhline(np.median(latency_us), color='tab:red', linestyle='--', label='median')
axes[0, 0].set(title='End-to-end latency by run', xlabel='Run', ylabel='Latency (us)')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

axes[0, 1].hist(latency_us, bins=min(20, max(5, MEASURED_RUNS // 5)), color='tab:blue', alpha=0.8)
axes[0, 1].axvline(np.percentile(latency_us, 95), color='tab:orange', linestyle='--', label='p95')
axes[0, 1].set(title='Latency distribution', xlabel='Latency (us)', ylabel='Count')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

axes[1, 0].plot(np.arange(1, MEASURED_RUNS + 1), gops, color='tab:green', linewidth=1)
axes[1, 0].axhline(gops.mean(), color='tab:red', linestyle='--', label='mean')
axes[1, 0].set(title='Effective compute throughput', xlabel='Run', ylabel='GOPS')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

parts = [control_us.mean(), c_recv_start_us.mean(), a_send_start_us.mean(), b_send_start_us.mean(), completion_us.mean()]
labels = ['AXI-Lite\nstart', 'C recv\nstart', 'A send\nstart', 'B send\nstart', 'Completion\nwait']
axes[1, 1].bar(labels, parts, color=['tab:purple', 'tab:blue', 'tab:orange', 'tab:green', 'tab:cyan'])
axes[1, 1].set(title='Mean host-side timing breakdown', ylabel='Time (us)')
axes[1, 1].grid(axis='y', alpha=0.25)

fig.suptitle('16x16 Q8.8 systolic accelerator — PS-observed performance', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(C_hw, cmap='coolwarm')
axes[0].set_title('Hardware result')
fig.colorbar(im0, ax=axes[0], shrink=0.8)
im1 = axes[1].imshow(error, cmap='magma')
axes[1].set_title(f'Absolute error (max={error.max():.5f})')
fig.colorbar(im1, ax=axes[1], shrink=0.8)
fig.tight_layout()
plt.show()

In [ ]:
a_buf.close()
b_buf.close()
c_buf.close()
print('Buffers released')